# 18 Summaries on real data: NIST StRD, airquality, the API schools, and R

The number words (`nums.find_average`, `find_standard_deviation`, `find_total`, `find_biggest`, `find_smallest`,
`find_percentile`, `count_each`), the table summaries with and without groups and `every=`, their intervals,
R's nine percentile rules, frequency and analytic weights, `find_summary_of`, `find_interval_of` (bootstrap),
`count_each(interval=)`, `find_percent_of_rows_where` and `find_correlation_between`.

**Only simplydata runs in this notebook.** The reference values (`references/18_summaries_real_data.csv`, from
`scripts/references/18_summaries_real_data.py`): the certified means and standard deviations of the nine NIST
StRD univariate datasets (LRE of at least 9 digits); base R (`sum`, `max`, `quantile(type = 1 ... 9)`, `table`,
`t.test`, `cor.test`, `ecdf` ...), DescTools, e1071, psych, WRS2, Hmisc, survey, boot and ppcor (simplydata calls
SciPy, statsmodels and NumPy for these words); SciPy's `quantile_test` and `bootstrap` with the same seed (the
parent); formulas written out (gamma and Somers' d from the pairs). With two columns `find_correlation_between`
gives back its p-value and interval with `show="result table"`, and with `allowing_for=` the partial correlation's
p-value (R ppcor's own, simplydata a5a8ff0 and later).

Data: the NIST StRD univariate files (`data/nist`), airquality (New York, 1973; Chambers et al. 1983), the API
stratified sample of R survey, the Hollander & Wolfe example of R's `cor.test` help page, and mtcars.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_numbers, read_row, read_rows, versions

NOTEBOOK = "18_summaries_real_data"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)
MONTHS = [5, 6, 7, 8, 9]


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def rows(word, *args, **options):
    return read_rows(quiet(word, *args, **options))

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Number words on the NIST StRD univariate datasets, and on airquality's ozone

In [2]:
for name in ["PiDigits", "Lottery", "Lew", "Mavro", "Michelso", "NumAcc1", "NumAcc2", "NumAcc3", "NumAcc4"]:
    nums = load(f"nist_{name}").get_column("y")
    ds = f"NIST {name}"
    checks.check(ds, "nums.find_average", "mean", quiet(nums.find_average))
    checks.check(ds, "nums.find_standard_deviation", "standard deviation", quiet(nums.find_standard_deviation))
    checks.check(ds, "nums.find_total", "total", quiet(nums.find_total))
    checks.check(ds, "nums.find_biggest", "biggest", quiet(nums.find_biggest))
    checks.check(ds, "nums.find_smallest", "smallest", quiet(nums.find_smallest))
    for percent in (10, 50, 90):
        checks.check(ds, "nums.find_percentile (percent=)", f"percentile {percent}", quiet(nums.find_percentile, percent))
    counted = rows(nums.count_each)
    count_column = list(counted[0])[1]
    checks.check_values(ds, "nums.count_each", {"different values": len(counted), "biggest count": counted[0][count_column]})

agrees  NIST PiDigits | nums.find_average | mean: simplydata 4.5348, NIST certified 4.5348; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/
agrees  NIST PiDigits | nums.find_standard_deviation | standard deviation: simplydata 2.86733906029, NIST certified 2.86733906029; LRE 15.0 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/
agrees  NIST PiDigits | nums.find_total | total: simplydata 22674, R sum 22674; relative difference 0.00e+00 (allowed 1e-12)
      Source: R sum (run reference) - https://search.r-project.org/R/refmans/stats/html/sum.html
agrees  NIST PiDigits | nums.find_biggest | biggest: simplydata 9, R max 9; relative difference 0.00e+00 (allowed 1e-12)
      Source: R max (run reference) - https://search.r-project.org/R/refmans/stats/html/max.html
agrees  NIST PiDigits | nums.find_smallest | smallest: simplydata 0, R min 0; relative difference 0.00e+00 (allowed 1e-12)
      So

In [3]:
airquality = load("airquality")
ozone = airquality.get_column("Ozone")
ozone.count_each()

How many times each number appears (68 different): (empty) (37 times), 23 (6 times), 13 (4 times), 14 (4 times), 16 (4 times), 18 (4 times), 20 (4 times), 21 (4 times) ... and 60 more.


,number,count,percent
1,NaN,37,24.2
2,23.0,6,3.9
3,13.0,4,2.6
4,14.0,4,2.6
5,16.0,4,2.6
...,...,...,...
64,115.0,1,0.7
65,118.0,1,0.7
66,122.0,1,0.7
67,135.0,1,0.7


In [4]:
ds = "airquality Ozone"
checks.check(ds, "nums.find_average", "mean", quiet(ozone.find_average))
checks.check(ds, "nums.find_standard_deviation", "standard deviation", quiet(ozone.find_standard_deviation))
checks.check(ds, "nums.find_total", "total", quiet(ozone.find_total))
checks.check(ds, "nums.find_biggest", "biggest", quiet(ozone.find_biggest))
checks.check(ds, "nums.find_smallest", "smallest", quiet(ozone.find_smallest))
checks.check(ds, "nums.find_percentile (percent=)", "percentile 25", quiet(ozone.find_percentile, 25))
counted = rows(ozone.count_each)
names = list(counted[0])
empty = [row for row in counted if row[names[0]] is None or row[names[0]] != row[names[0]]][0]
checks.check_values(ds, "nums.count_each", {"different values (empty counted)": len(counted), "empty cells": empty[names[1]],
                                            "empty cells percent (1 decimal)": empty[names[2]]})

agrees  airquality Ozone | nums.find_average | mean: simplydata 42.1293103448, R mean 42.1293103448; relative difference 1.69e-16 (allowed 1e-12)
      Source: R mean (run reference) - https://search.r-project.org/R/refmans/stats/html/mean.html
agrees  airquality Ozone | nums.find_standard_deviation | standard deviation: simplydata 32.9878845144, R sd 32.9878845144; relative difference 0.00e+00 (allowed 1e-10)
      Source: R sd (run reference) - https://search.r-project.org/R/refmans/stats/html/sd.html
agrees  airquality Ozone | nums.find_total | total: simplydata 4887, R sum 4887; relative difference 0.00e+00 (allowed 1e-12)
      Source: R sum (run reference) - https://search.r-project.org/R/refmans/stats/html/sum.html
agrees  airquality Ozone | nums.find_biggest | biggest: simplydata 168, R max 168; difference 0.00e+00 (allowed 0e+00)
      Source: R max (run reference) - https://search.r-project.org/R/refmans/stats/html/max.html
agrees  airquality Ozone | nums.find_smallest | smal

3

## Table summaries by group, with empty cells: airquality by month, and `every="month"` on the dates

In [5]:
air = load("air")
quiet(air.turn_into_dates, "date")
air.find_biggest_of("Ozone", for_each_group_in="Month")

Note: 37 empty cells in "Ozone" were skipped.
The biggest of "Ozone" for each "Month": 5.0 115, 6.0 71, 7.0 135, 8.0 168, 9.0 96.


,Month,biggest of Ozone
1,5.0,115.0
2,6.0,71.0
3,7.0,135.0
4,8.0,168.0
5,9.0,96.0


In [6]:
for every in (None, "month"):
    group = "date" if every else "Month"
    how = f'every="{every}"' if every else 'for_each_group_in="Month"'
    for label, word, key in [("find_biggest_of", air.find_biggest_of, "biggest"), ("find_smallest_of", air.find_smallest_of, "smallest"),
                             ("find_total_of", air.find_total_of, "total"), ("find_average_of", air.find_average_of, "mean"),
                             ("find_median_of", air.find_median_of, "median"),
                             ("find_standard_deviation_of", air.find_standard_deviation_of, "sd")]:
        found = rows(word, "Ozone", for_each_group_in=group, every=every)
        checks.check_values("airquality", f"{label} ({how})",
                            {f"Ozone {key}, month {m}": list(row.values())[-1] for m, row in zip(MONTHS, found)})
    found = rows(air.count_rows, for_each_group_in=group, every=every)
    checks.check_values("airquality", f"count_rows ({how})", {f"rows, month {m}": list(row.values())[-1] for m, row in zip(MONTHS, found)})
    found = rows(air.count_different_values_in, "Temp", for_each_group_in=group, every=every)
    checks.check_values("airquality", f"count_different_values_in ({how})",
                        {f"Temp values, month {m}": list(row.values())[-1] for m, row in zip(MONTHS, found)})
    values = {}
    for m, row in zip(MONTHS, rows(air.find_most_common_of, "Temp", for_each_group_in=group, every=every)):
        cells = list(row.values())
        values.update({f"Temp mode, month {m}": cells[1], f"Temp mode count, month {m}": cells[2],
                       f"Temp mode percent, month {m}": cells[3], f"Temp values tied at the top, month {m}": cells[4]})
    checks.check_values("airquality", f"find_most_common_of ({how})", values)
checks.check("airquality", "find_biggest_of", "Ozone biggest", quiet(air.find_biggest_of, "Ozone"))
checks.check("airquality", "find_smallest_of", "Ozone smallest", quiet(air.find_smallest_of, "Ozone"))
checks.check("airquality", "find_total_of", "Ozone total", quiet(air.find_total_of, "Ozone"))
checks.check("airquality", "count_rows", "rows", quiet(air.count_rows))
checks.check("airquality", "count_different_values_in", "Temp values", quiet(air.count_different_values_in, "Temp"))
checks.check("airquality", "find_most_common_of", "Temp mode", quiet(air.find_most_common_of, "Temp"))

agrees  airquality | find_biggest_of (for_each_group_in="Month") | Ozone biggest, month 5: simplydata 115, R tapply 115; relative difference 0.00e+00 (allowed 1e-12)
      Source: R tapply (run reference) - https://search.r-project.org/R/refmans/stats/html/tapply.html
agrees  airquality | find_biggest_of (for_each_group_in="Month") | Ozone biggest, month 6: simplydata 71, R tapply 71; relative difference 0.00e+00 (allowed 1e-12)
      Source: R tapply (run reference) - https://search.r-project.org/R/refmans/stats/html/tapply.html
agrees  airquality | find_biggest_of (for_each_group_in="Month") | Ozone biggest, month 7: simplydata 135, R tapply 135; relative difference 0.00e+00 (allowed 1e-12)
      Source: R tapply (run reference) - https://search.r-project.org/R/refmans/stats/html/tapply.html
agrees  airquality | find_biggest_of (for_each_group_in="Month") | Ozone biggest, month 8: simplydata 168, R tapply 168; relative difference 0.00e+00 (allowed 1e-12)
      Source: R tapply (run r

agrees  airquality | find_most_common_of (every="month") | Temp mode, month 5: simplydata 57, R table, smallest of the tied 57; difference 0.00e+00 (allowed 0e+00)
      Source: R table, smallest of the tied (run reference) - https://search.r-project.org/R/refmans/stats/html/table.html
agrees  airquality | find_most_common_of (every="month") | Temp mode count, month 5: simplydata 3, R table 3; difference 0.00e+00 (allowed 0e+00)
      Source: R table (run reference) - https://search.r-project.org/R/refmans/stats/html/table.html
agrees  airquality | find_most_common_of (every="month") | Temp mode percent, month 5: simplydata 9.67741935484, R table 9.67741935484; relative difference 0.00e+00 (allowed 1e-12)
      Source: R table (run reference) - https://search.r-project.org/R/refmans/stats/html/table.html
agrees  airquality | find_most_common_of (every="month") | Temp values tied at the top, month 5: simplydata 3, R table 3; difference 0.00e+00 (allowed 0e+00)
      Source: R table (run

1

## Intervals of the average, median, percentiles and standard deviation

In [7]:
def interval_values(found, prefix):
    values = {}
    for m, row in zip(MONTHS, found):
        for column in row:
            if "CI low" in column:
                values[f"{prefix} 95 % CI low, month {m}"] = row[column]
            elif "CI high" in column:
                values[f"{prefix} 95 % CI high, month {m}"] = row[column]
    return values


for label, word in [("find_average_of (interval=)", air.find_average_of), ("find_median_of (interval=)", air.find_median_of),
                    ("find_standard_deviation_of (interval=)", air.find_standard_deviation_of)]:
    checks.check_values("airquality", label, interval_values(rows(word, "Ozone", for_each_group_in="Month", interval=95), "Ozone"))
for percent in (25, 75):
    checks.check_values("airquality", "find_percentile_of (interval=)",
                        interval_values(rows(air.find_percentile_of, "Ozone", percent, for_each_group_in="Month", interval=95),
                                        f"Ozone p{percent}"))
found = rows(air.find_percentile_of, "Ozone", 90, interval=95)[0]
checks.check_values("airquality", "find_percentile_of (interval=)",
                    {f"Ozone p90 95 % CI {'low' if 'low' in c else 'high'}": v for c, v in found.items() if "CI" in c})

agrees  airquality | find_average_of (interval=) | Ozone 95 % CI low, month 5: simplydata 14.6387352112, R t.test 14.6387352112; relative difference 2.43e-16 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  airquality | find_average_of (interval=) | Ozone 95 % CI high, month 5: simplydata 32.5920340196, R t.test 32.5920340196; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  airquality | find_average_of (interval=) | Ozone 95 % CI low, month 6: simplydata 15.4486102671, R t.test 15.4486102671; relative difference 1.15e-16 (allowed 1e-10)
      Source: R t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  airquality | find_average_of (interval=) | Ozone 95 % CI high, month 6: simplydata 43.4402786218, R t.test 43.4402786218; relative difference 0.00e+00 (allowed 

agrees  airquality | find_standard_deviation_of (interval=) | Ozone 95 % CI low, month 5: simplydata 17.4296876046, chi-square formula in R 17.4296876046; relative difference 2.04e-16 (allowed 1e-10)
      Source: chi-square formula in R (run reference)
agrees  airquality | find_standard_deviation_of (interval=) | Ozone 95 % CI high, month 5: simplydata 30.6788257151, chi-square formula in R 30.6788257151; relative difference 0.00e+00 (allowed 1e-10)
      Source: chi-square formula in R (run reference)
agrees  airquality | find_standard_deviation_of (interval=) | Ozone 95 % CI low, month 6: simplydata 12.2986570185, chi-square formula in R 12.2986570185; relative difference 0.00e+00 (allowed 1e-10)
      Source: chi-square formula in R (run reference)
agrees  airquality | find_standard_deviation_of (interval=) | Ozone 95 % CI high, month 6: simplydata 34.8821728332, chi-square formula in R 34.8821728332; relative difference 2.04e-16 (allowed 1e-10)
      Source: chi-square formula in 

2

## Percentiles: R's nine rules, frequency weights (the daily temperatures as a frequency table) and analytic
weights (the API schools' sampling weights)

In [8]:
RULES = {1: "inverted cdf", 2: "averaged inverted cdf", 3: "closest observation", 4: "interpolated inverted cdf",
         5: "hazen", 6: "weibull", 7: "linear", 8: "median unbiased", 9: "normal unbiased"}
for column in ("Ozone", "Wind"):
    for name in RULES.values():
        checks.check_values(f"airquality {column}", f'find_percentile_of (test_method="{name}")',
                            {f"percentile {p}": quiet(air.find_percentile_of, column, p, test_method=name) for p in (5, 25, 50, 80, 99)})
    checks.check(f"airquality {column}", 'find_percentile_of (test_method="inverted_cdf")', "percentile 90",
                 quiet(air.find_percentile_of, column, 90, test_method="inverted_cdf"))

frequency = load("frequency_rows")
ds = "airquality Temp as a frequency table"
for p in (10, 90):
    checks.check(ds, 'find_percentile_of (weights=, weight_kind="frequency")', f"percentile {p}",
                 quiet(frequency.find_percentile_of, "temperature", p, weights="days", weight_kind="frequency"))
checks.check(ds, 'find_median_of (weights=, weight_kind="frequency")', "median",
             quiet(frequency.find_median_of, "temperature", weights="days", weight_kind="frequency"))
checks.check(ds, "find_average_of (weights=)", "average", quiet(frequency.find_average_of, "temperature", weights="days"))
checks.check(ds, 'find_standard_deviation_of (weights=, weight_kind="frequency")', "standard deviation",
             quiet(frequency.find_standard_deviation_of, "temperature", weights="days", weight_kind="frequency"))

schools = load("schools_rows")
ds = "API stratified sample (apistrat), api00"
for p in (25, 50, 90):
    checks.check(ds, 'find_percentile_of (weights=, weight_kind="analytic")', f"percentile {p}",
                 quiet(schools.find_percentile_of, "api00", p, weights="pw", weight_kind="analytic"))
checks.check(ds, 'find_median_of (weights=, weight_kind="analytic")', "median",
             quiet(schools.find_median_of, "api00", weights="pw", weight_kind="analytic"))
checks.check(ds, 'find_standard_deviation_of (weights=, weight_kind="analytic")', "standard deviation",
             quiet(schools.find_standard_deviation_of, "api00", weights="pw", weight_kind="analytic"))
checks.check(ds, "find_average_of (weights=)", "average", quiet(schools.find_average_of, "api00", weights="pw"))

agrees  airquality Ozone | find_percentile_of (test_method="inverted cdf") | percentile 5: simplydata 7, R quantile(type = 1) 7; relative difference 0.00e+00 (allowed 1e-12)
      Source: R quantile(type = 1) (run reference) - https://search.r-project.org/R/refmans/stats/html/quantile.html
agrees  airquality Ozone | find_percentile_of (test_method="inverted cdf") | percentile 25: simplydata 18, R quantile(type = 1) 18; relative difference 0.00e+00 (allowed 1e-12)
      Source: R quantile(type = 1) (run reference) - https://search.r-project.org/R/refmans/stats/html/quantile.html
agrees  airquality Ozone | find_percentile_of (test_method="inverted cdf") | percentile 50: simplydata 31, R quantile(type = 1) 31; relative difference 0.00e+00 (allowed 1e-12)
      Source: R quantile(type = 1) (run reference) - https://search.r-project.org/R/refmans/stats/html/quantile.html
agrees  airquality Ozone | find_percentile_of (test_method="inverted cdf") | percentile 80: simplydata 73, R quantile(typ

2

## `find_summary_of`: every measure, by month and by `every="month"`

In [9]:
SHOW = ["count", "average", "median", "total", "smallest", "biggest", "standard deviation", "variance",
        "standard error", "confidence interval", "range", "interquartile range", "empty cells", "geometric mean",
        "skewness", "kurtosis", "coefficient of variation", "median absolute deviation", "trimmed average",
        "trimmed average 20", "most common", "percentile 90", "harmonic mean", "winsorised average",
        "winsorised average 10"]
for every in (None, "month"):
    how = 'every="month"' if every else 'for_each_group_in="Month"'
    values = {}
    for m, row in zip(MONTHS, rows(air.find_summary_of, "Ozone", for_each_group_in="date" if every else "Month",
                                   every=every, show=SHOW)):
        for column, value in row.items():
            if "95 % CI low" in column:
                values[f"Ozone 95 % CI low, month {m}"] = value
            elif "95 % CI high" in column:
                values[f"Ozone 95 % CI high, month {m}"] = value
            else:
                values[f"Ozone {column}, month {m}"] = value
    checks.check_values("airquality", f"find_summary_of ({how}, show=)", values)

agrees  airquality | find_summary_of (for_each_group_in="Month", show=) | Ozone count, month 5: simplydata 26, R (e1071, psych, WRS2, base) 26; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, base) (run reference)
agrees  airquality | find_summary_of (for_each_group_in="Month", show=) | Ozone average, month 5: simplydata 23.6153846154, R (e1071, psych, WRS2, base) 23.6153846154; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, base) (run reference)
agrees  airquality | find_summary_of (for_each_group_in="Month", show=) | Ozone median, month 5: simplydata 18, R (e1071, psych, WRS2, base) 18; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, base) (run reference)
agrees  airquality | find_summary_of (for_each_group_in="Month", show=) | Ozone total, month 5: simplydata 614, R (e1071, psych, WRS2, base) 614; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, bas

agrees  airquality | find_summary_of (every="month", show=) | Ozone count, month 5: simplydata 26, R (e1071, psych, WRS2, base) 26; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, base) (run reference)
agrees  airquality | find_summary_of (every="month", show=) | Ozone average, month 5: simplydata 23.6153846154, R (e1071, psych, WRS2, base) 23.6153846154; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, base) (run reference)
agrees  airquality | find_summary_of (every="month", show=) | Ozone median, month 5: simplydata 18, R (e1071, psych, WRS2, base) 18; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, base) (run reference)
agrees  airquality | find_summary_of (every="month", show=) | Ozone total, month 5: simplydata 614, R (e1071, psych, WRS2, base) 614; relative difference 0.00e+00 (allowed 1e-12)
      Source: R (e1071, psych, WRS2, base) (run reference)
agrees  airquality | find_sum

## `find_interval_of`: bootstrap intervals (SciPy with the same seed; R `boot.ci` with 200 000 resamples within
4 Monte Carlo standard errors)

In [10]:
for summary_name in ("average", "median", "standard deviation"):
    for method in ("bootstrap", "percentile bootstrap", "reverse percentile bootstrap"):
        found = rows(air.find_interval_of, "Ozone", summary_name, test_method=method, resamples=9999, with_seed=1)[0]
        checks.check_values("airquality Ozone", f'find_interval_of ("{summary_name}", test_method="{method}")',
                            {f"95 % CI {'low' if 'low' in c else 'high'}": v for c, v in found.items() if "CI" in c})

agrees  airquality Ozone | find_interval_of ("average", test_method="bootstrap") | 95 % CI low: simplydata 36.7277760949, SciPy bootstrap, same seed 36.7277760949; relative difference 1.93e-16 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("average", test_method="bootstrap") | 95 % CI low: simplydata 36.7277760949, R boot.ci, 200 000 resamples (4 MCSE) 36.6034482759; difference 1.24e-01 (allowed 3e-01)
      Source: R boot.ci, 200 000 resamples (4 MCSE) (run reference)
agrees  airquality Ozone | find_interval_of ("average", test_method="bootstrap") | 95 % CI high: simplydata 48.7178164094, SciPy bootstrap, same seed 48.7178164094; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  ai

agrees  airquality Ozone | find_interval_of ("average", test_method="percentile bootstrap") | 95 % CI low: simplydata 36.3702586207, SciPy bootstrap, same seed 36.3702586207; relative difference 1.95e-16 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("average", test_method="percentile bootstrap") | 95 % CI low: simplydata 36.3702586207, R boot.ci, 200 000 resamples (4 MCSE) 36.3103448276; difference 5.99e-02 (allowed 3e-01)
      Source: R boot.ci, 200 000 resamples (4 MCSE) (run reference)
agrees  airquality Ozone | find_interval_of ("average", test_method="percentile bootstrap") | 95 % CI high: simplydata 48.1810344828, SciPy bootstrap, same seed 48.1810344828; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scip

agrees  airquality Ozone | find_interval_of ("average", test_method="reverse percentile bootstrap") | 95 % CI low: simplydata 36.0775862069, SciPy bootstrap, same seed 36.0775862069; relative difference 1.97e-16 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("average", test_method="reverse percentile bootstrap") | 95 % CI low: simplydata 36.0775862069, R boot.ci, 200 000 resamples (4 MCSE) 36; difference 7.76e-02 (allowed 4e-01)
      Source: R boot.ci, 200 000 resamples (4 MCSE) (run reference)
agrees  airquality Ozone | find_interval_of ("average", test_method="reverse percentile bootstrap") | 95 % CI high: simplydata 47.888362069, SciPy bootstrap, same seed 47.888362069; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/gen

agrees  airquality Ozone | find_interval_of ("median", test_method="bootstrap") | 95 % CI low: simplydata 23.5, SciPy bootstrap, same seed 23.5; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("median", test_method="bootstrap") | 95 % CI high: simplydata 39, SciPy bootstrap, same seed 39; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html


agrees  airquality Ozone | find_interval_of ("median", test_method="percentile bootstrap") | 95 % CI low: simplydata 23, SciPy bootstrap, same seed 23; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("median", test_method="percentile bootstrap") | 95 % CI high: simplydata 38.5, SciPy bootstrap, same seed 38.5; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html


agrees  airquality Ozone | find_interval_of ("median", test_method="reverse percentile bootstrap") | 95 % CI low: simplydata 24.5, SciPy bootstrap, same seed 24.5; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("median", test_method="reverse percentile bootstrap") | 95 % CI high: simplydata 40, SciPy bootstrap, same seed 40; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html


agrees  airquality Ozone | find_interval_of ("standard deviation", test_method="bootstrap") | 95 % CI low: simplydata 28.7772455772, SciPy bootstrap, same seed 28.7772455772; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("standard deviation", test_method="bootstrap") | 95 % CI high: simplydata 39.8293431277, SciPy bootstrap, same seed 39.8293431277; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html


agrees  airquality Ozone | find_interval_of ("standard deviation", test_method="percentile bootstrap") | 95 % CI low: simplydata 27.600315799, SciPy bootstrap, same seed 27.600315799; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("standard deviation", test_method="percentile bootstrap") | 95 % CI high: simplydata 38.0662688047, SciPy bootstrap, same seed 38.0662688047; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html


agrees  airquality Ozone | find_interval_of ("standard deviation", test_method="reverse percentile bootstrap") | 95 % CI low: simplydata 27.9095002242, SciPy bootstrap, same seed 27.9095002242; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  airquality Ozone | find_interval_of ("standard deviation", test_method="reverse percentile bootstrap") | 95 % CI high: simplydata 38.3754532299, SciPy bootstrap, same seed 38.3754532299; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy bootstrap, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html


## `count_each(interval=)` and `find_percent_of_rows_where`

In [11]:
measured = load("measured_rows")
values = {}
for row in rows(measured.count_each, "month", interval=95):
    m = row["month"].split(" ")[1]
    for column, value in row.items():
        if "low" in column:
            values[f"month {m} 95 % low"] = value
        elif "high" in column:
            values[f"month {m} 95 % high"] = value
checks.check_values("airquality, days with ozone", "count_each (interval=)", values)

for every in (None, "month"):
    how = 'every="month"' if every else 'for_each_group_in="Month"'
    for interval_by in ("exact", "wilson", "agresti-coull"):
        values = {}
        for m, row in zip(MONTHS, rows(air.find_percent_of_rows_where, "Ozone", is_above=80,
                                       for_each_group_in="date" if every else "Month", every=every, interval_by=interval_by)):
            for column, value in row.items():
                if "low" in column:
                    values[f"95 % CI low, month {m}"] = value
                elif "high" in column:
                    values[f"95 % CI high, month {m}"] = value
                elif "percent" in column:
                    values[f"percent of days above 80, month {m}"] = value
        checks.check_values("airquality", f'find_percent_of_rows_where ({how}, interval_by="{interval_by}")', values)

agrees  airquality, days with ozone | count_each (interval=) | month 9 95 % low: simplydata 16.3793103448, R DescTools MultinomCI("sisonglaz") 16.3793103448; relative difference 0.00e+00 (allowed 1e-06)


      Source: R DescTools MultinomCI("sisonglaz") (run reference) - https://search.r-project.org/CRAN/refmans/DescTools/html/MultinomCI.html (version 0.99.60)
agrees  airquality, days with ozone | count_each (interval=) | month 9 95 % low: simplydata 16.3793103448, statsmodels multinomial_proportions_confint 16.3793103448; relative difference 0.00e+00 (allowed 1e-12)
      Source: statsmodels multinomial_proportions_confint (run reference)
agrees  airquality, days with ozone | count_each (interval=) | month 9 95 % high: simplydata 34.8937460448, R DescTools MultinomCI("sisonglaz") 34.8937460448; relative difference 4.46e-14 (allowed 1e-06)
      Source: R DescTools MultinomCI("sisonglaz") (run reference) - https://search.r-project.org/CRAN/refmans/DescTools/html/MultinomCI.html (version 0.99.60)
agrees  airquality, days with ozone | count_each (interval=) | month 5 95 % low: simplydata 13.7931034483, R DescTools MultinomCI("sisonglaz") 13.7931034483; relative difference 1.29e-16 (allo

## `find_correlation_between`

r on the Hollander & Wolfe example of R's `cor.test` help page and on mtcars; every pair's r and p-value with
`show="each pair"`; gamma and Somers' d; the partial correlation; the p-value table.

In [12]:
hollander = load("hollander_rows")
cars = load("mtcars")
for name, table, a, b in [("Hollander & Wolfe (R cor.test example)", hollander, "x", "y"), ("mtcars mpg, wt", cars, "mpg", "wt")]:
    for method in ("pearson", "spearman", "kendall"):
        checks.check(name, f'find_correlation_between (test_method="{method}")', "r",
                     quiet(table.find_correlation_between, a, b, test_method=method))
for method in ("pearson", "spearman", "kendall"):
    values = {}
    for row in rows(cars.find_correlation_between, "mpg", "wt", "hp", "qsec", test_method=method, show="each pair"):
        a, b = list(row.values())[:2]
        values[f"r {a}, {b}"] = row["r"]
        values[f"p-value {a}, {b}"] = row["p-value"]
    checks.check_values("mtcars", f'find_correlation_between (test_method="{method}", show="each pair")', values)
for name, table, a, b in [("mtcars cyl, gear", cars, "cyl", "gear"), ("mtcars carb, am", cars, "carb", "am"),
                          ("airquality Month, Temp", air, "Month", "Temp")]:
    for method in ("gamma", "somers"):
        checks.check(name, f'find_correlation_between (test_method="{method}")', "estimate",
                     quiet(table.find_correlation_between, a, b, test_method=method))
checks.check("mtcars mpg, wt | hp", "find_correlation_between (allowing_for=)", "partial r",
             quiet(cars.find_correlation_between, "mpg", "wt", allowing_for="hp"))
table = quiet(cars.find_correlation_between, "mpg", "wt", "hp", show="p-value")
first = table.get_column_names()[0]
p_values = {row[first]: row for row in read_rows(table)}
checks.check_values("mtcars", 'find_correlation_between (show="p-value")',
                    {f"p-value {a}, {b}": p_values[a][b] for a, b in [("mpg", "wt"), ("mpg", "hp"), ("wt", "hp")]})

agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="pearson") | r: simplydata 0.571181557564, R cor.test 0.571181557564; relative difference 0.00e+00 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="spearman") | r: simplydata 0.6, R cor.test 0.6; relative difference 1.85e-16 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="kendall") | r: simplydata 0.444444444444, R cor.test 0.444444444444; relative difference 1.25e-16 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  mtcars mpg, wt | find_correlation_between (test_method="pearson") | r: simplydata -0.8

agrees  mtcars mpg, wt | find_correlation_between (test_method="kendall") | r: simplydata -0.727832149528, R cor.test -0.727832149528; relative difference 1.53e-16 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  mtcars | find_correlation_between (test_method="pearson", show="each pair") | r mpg, wt: simplydata -0.867659376517, R cor.test -0.867659376517; relative difference 0.00e+00 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  mtcars | find_correlation_between (test_method="pearson", show="each pair") | p-value mpg, wt: simplydata 1.29395870135e-10, R cor.test 1.29395870135e-10; relative difference 2.80e-15 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  mtcars | find_correlation_between (test_method="pearson", show="each pair") | r mpg, 

agrees  mtcars cyl, gear | find_correlation_between (test_method="gamma") | estimate: simplydata -0.657370517928, pairs counted in R -0.657370517928; relative difference 0.00e+00 (allowed 1e-10)
      Source: pairs counted in R (run reference)
agrees  mtcars cyl, gear | find_correlation_between (test_method="gamma") | estimate: simplydata -0.657370517928, R DescTools -0.657370517928; relative difference 0.00e+00 (allowed 1e-10)
      Source: R DescTools (run reference)
agrees  mtcars cyl, gear | find_correlation_between (test_method="somers") | estimate: simplydata -0.501519756839, pairs counted in R -0.501519756839; relative difference 0.00e+00 (allowed 1e-10)
      Source: pairs counted in R (run reference)
agrees  mtcars cyl, gear | find_correlation_between (test_method="somers") | estimate: simplydata -0.501519756839, R DescTools -0.501519756839; relative difference 0.00e+00 (allowed 1e-10)
      Source: R DescTools (run reference)


agrees  mtcars carb, am | find_correlation_between (test_method="gamma") | estimate: simplydata -0.0947368421053, pairs counted in R -0.0947368421053; relative difference 5.86e-16 (allowed 1e-10)
      Source: pairs counted in R (run reference)
agrees  mtcars carb, am | find_correlation_between (test_method="gamma") | estimate: simplydata -0.0947368421053, R DescTools -0.0947368421053; relative difference 5.86e-16 (allowed 1e-10)
      Source: R DescTools (run reference)
agrees  mtcars carb, am | find_correlation_between (test_method="somers") | estimate: simplydata -0.0471204188482, pairs counted in R -0.0471204188482; relative difference 8.84e-16 (allowed 1e-10)
      Source: pairs counted in R (run reference)
agrees  mtcars carb, am | find_correlation_between (test_method="somers") | estimate: simplydata -0.0471204188482, R DescTools -0.0471204188482; relative difference 8.84e-16 (allowed 1e-10)
      Source: R DescTools (run reference)
agrees  airquality Month, Temp | find_correlat

3

### The two-column p-values and intervals: `show="result table"`

With two columns, `find_correlation_between(a, b, show="result table")` gives back a one-row table: r, its 95 %
interval, the p-value, which directions were tested and the rows used. Its p-value is compared with R `cor.test`
(same `method` and `alternative`; for Spearman without ties R's default exact p-value, which simplydata takes from
R; with ties `exact = FALSE`) to a relative difference of 1e-10; the shuffled p-value with SciPy `permutation_test`
with the same seed (1e-12) and with a million shuffles (4 Monte Carlo standard errors); the bootstrap interval with
SciPy `bootstrap` (BCa, same seed, 1e-10). With `allowing_for=` the same table gives the partial correlation's
p-value, compared with R ppcor `pcor.test` and with the t test of the two sets of R `lm` residuals on n - 3 degrees of
freedom (1e-10).

In [13]:
SIDES = [{}, {"sides": 1, "higher": "positive"}, {"sides": 1, "higher": "negative"}, {"alternative": "greater"}]
for name, table, a, b in [("Hollander & Wolfe (R cor.test example)", hollander, "x", "y"), ("mtcars mpg, wt", cars, "mpg", "wt")]:
    for method in ("pearson", "spearman", "kendall"):
        for options in SIDES:
            found = read_row(quiet(table.find_correlation_between, a, b, test_method=method, show="result table", **options))
            option_text = ", ".join(f"{k}={v!r}" for k, v in options.items())
            label = f'find_correlation_between (test_method="{method}"{", " + option_text if option_text else ""})'
            checks.check(name, label, "p-value", found["p-value"])
ds = "Hollander & Wolfe (R cor.test example)"
found = read_row(quiet(hollander.find_correlation_between, "x", "y", p_value_by="shuffling", shuffles=9999, with_seed=1,
                       show="result table"))
checks.check(ds, 'find_correlation_between (p_value_by="shuffling", shuffles=)', "p-value", found["p-value"])
found = read_row(quiet(hollander.find_correlation_between, "x", "y", interval_by="bootstrap", resamples=9999, with_seed=1,
                       show="result table"))
checks.check_values(ds, 'find_correlation_between (interval_by="bootstrap", resamples=)',
                    {"95 % CI low": found["95 % CI low"], "95 % CI high": found["95 % CI high"]})
found = read_row(quiet(cars.find_correlation_between, "mpg", "wt", allowing_for="hp", show="result table"))
checks.check("mtcars mpg, wt | hp", "find_correlation_between (allowing_for=)", "p-value", found["p-value"])

agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="pearson") | p-value: simplydata 0.108173054349, R cor.test 0.108173054349; relative difference 2.69e-15 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="pearson", sides=1, higher='positive') | p-value: simplydata 0.0540865271747, R cor.test 0.0540865271747; relative difference 3.59e-15 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="pearson", sides=1, higher='negative') | p-value: simplydata 0.945913472825, R cor.test 0.945913472825; relative difference 2.35e-16 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.ht

agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="spearman", sides=1, higher='positive') | p-value: simplydata 0.0483989197531, R cor.test (default: exact) 0.0483989197531; relative difference 4.30e-16 (allowed 1e-10)
      Source: R cor.test (default: exact) (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="spearman", sides=1, higher='negative') | p-value: simplydata 0.957137345679, R cor.test (default: exact) 0.957137345679; relative difference 1.16e-16 (allowed 1e-10)
      Source: R cor.test (default: exact) (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (test_method="spearman", alternative='greater') | p-value: simplydata 0.0483989197531, R cor.test (default: exact) 0.0483989197531; relative difference 4.30e-16 (allowe

agrees  mtcars mpg, wt | find_correlation_between (test_method="spearman", alternative='greater') | p-value: simplydata 0.999999999993, R cor.test (exact = FALSE) 0.999999999993; relative difference 1.11e-16 (allowed 1e-10)
      Source: R cor.test (exact = FALSE) (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  mtcars mpg, wt | find_correlation_between (test_method="kendall") | p-value: simplydata 6.7057704056e-09, R cor.test 6.7057704056e-09; relative difference 4.93e-16 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html


agrees  mtcars mpg, wt | find_correlation_between (test_method="kendall", sides=1, higher='positive') | p-value: simplydata 0.999999996647, R cor.test 0.999999996647; relative difference 0.00e+00 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  mtcars mpg, wt | find_correlation_between (test_method="kendall", sides=1, higher='negative') | p-value: simplydata 3.3528852028e-09, R cor.test 3.3528852028e-09; relative difference 3.70e-16 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html
agrees  mtcars mpg, wt | find_correlation_between (test_method="kendall", alternative='greater') | p-value: simplydata 0.999999996647, R cor.test 0.999999996647; relative difference 0.00e+00 (allowed 1e-10)
      Source: R cor.test (run reference) - https://search.r-project.org/R/refmans/stats/html/cor.test.html


agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (p_value_by="shuffling", shuffles=) | p-value: simplydata 0.126, SciPy permutation_test, same seed 0.126; relative difference 0.00e+00 (allowed 1e-12)
      Source: SciPy permutation_test, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.permutation_test.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (p_value_by="shuffling", shuffles=) | p-value: simplydata 0.126, 1 000 000 shuffles, other seed (4 MCSE) 0.12671957672; difference 7.20e-04 (allowed 1e-02)
      Source: 1 000 000 shuffles, other seed (4 MCSE) (run reference)


agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (interval_by="bootstrap", resamples=) | 95 % CI low: simplydata -0.418066616454, SciPy bootstrap BCa, same seed -0.418066616454; relative difference 2.66e-16 (allowed 1e-10)
      Source: SciPy bootstrap BCa, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  Hollander & Wolfe (R cor.test example) | find_correlation_between (interval_by="bootstrap", resamples=) | 95 % CI high: simplydata 0.917441898579, SciPy bootstrap BCa, same seed 0.917441898579; relative difference 1.21e-16 (allowed 1e-10)
      Source: SciPy bootstrap BCa, same seed (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.bootstrap.html
agrees  mtcars mpg, wt | hp | find_correlation_between (allowing_for=) | p-value: simplydata 1.1196471362e-06, residuals of R lm, t on n - 3 1.1196471362e-06; relative difference 1.63e-14 (allowed 1e-10)
 

2

## Result

In [14]:
checks.finish()

18_summaries_real_data: 842 comparisons - 831 agrees, 10 explained difference – reference, 1 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,18_summaries_real_data,NIST PiDigits,nums.find_average,mean,4.534800,4.534800,NIST certified,0.000000e+00,9.000000e+00,lre,1.500000e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,18_summaries_real_data,NIST PiDigits,nums.find_standard_deviation,standard deviation,2.867339,2.867339,NIST certified,-1.332268e-15,9.000000e+00,lre,1.500000e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,18_summaries_real_data,NIST PiDigits,nums.find_total,total,22674.000000,22674.000000,R sum,0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,18_summaries_real_data,NIST PiDigits,nums.find_biggest,biggest,9.000000,9.000000,R max,0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,18_summaries_real_data,NIST PiDigits,nums.find_smallest,smallest,0.000000,0.000000,R min,0.000000e+00,1.000000e-12,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
837,18_summaries_real_data,Hollander & Wolfe (R cor.test example),"find_correlation_between (p_value_by=""shufflin...",p-value,0.126000,0.126720,"1 000 000 shuffles, other seed (4 MCSE)",-7.195767e-04,1.337337e-02,absolute,7.195767e-04,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
838,18_summaries_real_data,Hollander & Wolfe (R cor.test example),"find_correlation_between (interval_by=""bootstr...",95 % CI low,-0.418067,-0.418067,"SciPy bootstrap BCa, same seed",-1.110223e-16,1.000000e-10,relative,2.655613e-16,PASS,,agrees,,,,run reference,https://docs.scipy.org/doc/scipy-1.18.1/refere...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
839,18_summaries_real_data,Hollander & Wolfe (R cor.test example),"find_correlation_between (interval_by=""bootstr...",95 % CI high,0.917442,0.917442,"SciPy bootstrap BCa, same seed",1.110223e-16,1.000000e-10,relative,1.210129e-16,PASS,,agrees,,,,run reference,https://docs.scipy.org/doc/scipy-1.18.1/refere...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
840,18_summaries_real_data,"mtcars mpg, wt | hp",find_correlation_between (allowing_for=),p-value,0.000001,0.000001,"residuals of R lm, t on n - 3",-1.821121e-20,1.000000e-10,relative,1.626513e-14,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
